# Lab 1 — Build the triage crew

**~25 minutes · nothing to fill in**

Earlier in the course you built an agent as a graph. You wrote every step and every edge yourself.

This lab builds the same kind of agent in a different way, with **CrewAI**. You describe **who** is on
the team and **what** each member does. CrewAI then decides the order of the model calls.

The example is the **Global Bank customer support desk**. Bank customers send in tickets. The desk
reads each ticket and sends it to the team that owns it. Then it writes the first reply to the customer.

There are no `___` blanks and no score. Run the cells in order. Read what comes back, and answer the
questions in your head as you go.

**Words used in this lab**

- **Agent:** a model with a job description. It can also call tools.
- **Tool:** a Python function that the model can ask to run, for example to look up a ticket.
- **Task:** one piece of work for one agent. It says what to do and what the answer must look like.
- **Crew:** a group of agents and the tasks they run. This is CrewAI's word.
- **Request:** one call to the model. One task can need more than one request.
- **Token:** the unit a model reads and writes. A token is roughly three quarters of an English word.
  **Prompt** tokens are what the model reads. **Completion** tokens are what it writes.

## 1 · The model

Every call goes to the course's model gateway. The gateway is one address that serves several models.
Your sandbox already has three settings for it: the model name, the gateway address and your key. You
do not paste a key anywhere.

**CrewAI takes the plain model name.** You do not add an `openai/` prefix. CrewAI 1.x connects to
OpenAI-style APIs directly, so it does not need one. Google ADK, in Lab 3, is the opposite. If you copy
this cell into an ADK notebook, it fails.

In [ ]:
import os
from crewai import LLM

llm = LLM(
    model=os.environ["OPENAI_MODEL"],        # plain name - no "openai/" prefix
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)
print("model:", llm.model)
print("provider:", llm.provider)      # "openai" is the default

## 2 · A tool

An agent can only work with what it can reach. The dictionary below stands in for the bank's ticket
system. It holds four customer tickets, and the names of the four teams on the desk.

The docstring matters. The model reads it to decide whether to call this tool.

In [ ]:
TICKETS = {
    "GB-T-4471": "My transfer of Rs 25,000 to my landlord failed twice, but my account was debited once.",
    "GB-T-4472": "I get 'invalid OTP' every time I log in on my new phone, since yesterday.",
    "GB-T-4473": "Please update my registered address. I have moved to Pune.",
    "GB-T-4474": "My savings account shows Rs 1,200 less than my passbook.",
}

# The four teams on the desk. They match the Global Bank services from Day 1.
TEAMS = ["Accounts", "Transactions", "Authentication", "Customer"]
TEAM_RULE = ("The category is the kind of problem, in a few words. "
             "The team must be one of: " + ", ".join(TEAMS) + ".")

from crewai.tools import tool

@tool("ticket_lookup")
def ticket_lookup(ticket_id: str) -> str:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return TICKETS.get(ticket_id, f"No ticket found with id {ticket_id}")

print(ticket_lookup.name)

## 3 · One agent, one task

In CrewAI an **agent** is three strings: a **role**, a **goal** and a **backstory**. CrewAI turns them
into the system prompt. The system prompt is the set of instructions the model reads before the work
itself.

A **task** has a `description` and an `expected_output`. The `expected_output` is more important than
it looks. The next task in the crew relies on it, so treat it as a contract.

`allow_delegation=False` stops this agent from passing its work to another agent. Delegation is one of
the failure modes at the end of Lab 2.

> **Why `await crew.kickoff_async()` and not `crew.kickoff()`?** A notebook already runs an event loop.
> CrewAI 1.x refuses to start a synchronous run inside a running loop. It raises
> `RuntimeError: Agent execution was invoked synchronously from within a running event loop`.
> So in a notebook, use **`await crew.kickoff_async()`**. In a plain `.py` script, `crew.kickoff()` is
> correct. Most CrewAI examples on the web use `kickoff()`, so watch for this when you copy one into
> Jupyter.

In [ ]:
from crewai import Agent, Task, Crew, Process

triage = Agent(
    role="Support Triage Analyst",
    goal="Classify an incoming ticket and name the team that owns it",
    backstory="You triage the Global Bank customer support desk. You always send a ticket to the correct team.",
    llm=llm,
    tools=[ticket_lookup],
    allow_delegation=False,
    verbose=False,
)

classify = Task(
    description=f"Look up ticket GB-T-4471 and classify it. Give the category and the owning team. {TEAM_RULE}",
    expected_output="Exactly two lines - 'Category: <x>' then 'Team: <y>'",
    agent=triage,
)

crew = Crew(agents=[triage], tasks=[classify], process=Process.sequential, verbose=False)
result = await crew.kickoff_async()
print(result)

**Look at what you did not write.** You wrote no graph, no edges, no state object and no routing
function. You named a role and described a task. This is the main trade-off in CrewAI: you write less,
and the framework decides more. Lab 2 measures what that costs.

## 4 · Read the usage

`usage_metrics` is on the crew, not on the result. Look at `successful_requests`. One task needed more
than one request. The tool call is one round trip to the model, and the final answer is another.

This number is correct for a crew with **one** agent. Section 5 shows where it goes wrong.

In [ ]:
print(crew.usage_metrics)

## 5 · Three agents, in order

Now a real crew with three agents. `Process.sequential` runs the tasks one after another, in list
order. Each task's output becomes context for the next task. You still do not write the wiring. The
order of the `tasks` list is the wiring.

Watch the ticket text travel from the first task to the third. You do not pass it anywhere yourself.

In [ ]:
researcher = Agent(
    role="Ticket Researcher",
    goal="Retrieve the ticket and state the facts in it, without interpreting them",
    backstory="You pull the raw ticket and never guess beyond what it says.",
    llm=llm, tools=[ticket_lookup], allow_delegation=False,
)

classifier = Agent(
    role="Support Triage Analyst",
    goal="Classify a ticket and name the owning team",
    backstory="You triage the Global Bank customer support desk.",
    llm=llm, allow_delegation=False,
)

writer = Agent(
    role="Response Drafter",
    goal="Write the first reply the bank customer will read",
    backstory="You write to Global Bank customers. You write plainly, promise only what the team can do, "
              "and never invent a timeline.",
    llm=llm, allow_delegation=False,
)

t1 = Task(description="Retrieve ticket GB-T-4471 and list the facts it contains.",
          expected_output="A short bulleted list of facts, no interpretation.", agent=researcher)
t2 = Task(description=f"Classify that ticket and name the owning team. {TEAM_RULE}",
          expected_output="'Category: <x>' and 'Team: <y>' on separate lines.", agent=classifier)
t3 = Task(description="Draft a three-sentence first reply to the customer.",
          expected_output="Three sentences, no invented timeline.", agent=writer)

desk = Crew(agents=[researcher, classifier, writer], tasks=[t1, t2, t3],
            process=Process.sequential, verbose=False)

before = llm.get_token_usage_summary()          # take a snapshot of the model's usage first
out = await desk.kickoff_async()
sequential = llm.get_token_usage_summary().delta_since(before)

print(out)
print()
print("crew.usage_metrics:", desk.usage_metrics)
print("what this run made:", sequential)

**One run, two different counts.** The second line is what the gateway actually served. The first
line is several times bigger. Here is why:

- `crew.usage_metrics` adds up each agent's `llm` usage, once per agent.
- An `LLM` object keeps a running total of its usage.
- All three agents share `llm`, so CrewAI counts the same total three times.
- That total also still includes the run from section 3.

So the rule for this lab and Lab 2 is: **take a snapshot of the model object before `kickoff`, and
call `delta_since` after it.** Of the counts here, only this one matches what the gateway served.

## 6 · Let a manager decide the order

`Process.hierarchical` replaces your task order with a **manager**. The manager is one more model
(LLM) that decides which agent does what, and when. From here on, the framework makes decisions for
you.

It needs its own `manager_llm`. Run it, then compare the numbers with the sequential run above.

In [ ]:
managed = Crew(
    agents=[researcher, classifier, writer],
    tasks=[t1, t2, t3],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=False,
)
before = llm.get_token_usage_summary()          # the manager also uses llm, so it is counted too
out2 = await managed.kickoff_async()
hierarchical = llm.get_token_usage_summary().delta_since(before)

print(out2)
print()
print("sequential  :", sequential)
print("hierarchical:", hierarchical)

### What to take away

Compare the two lines. The hierarchical run added a manager that can change the plan. You paid for it
in requests and tokens. On this ticket, the order of the three tasks was already clear.

So **add a manager only when you can say why you need one.** Start with sequential.

---

**Next:** Lab 2 adds a critic agent, and then moves the workers to a different model. You decide from
the numbers whether either change is worth it.